# 🔄 Topic 06: CI/CD for Machine Learning (Continuous Integration/Continuous Delivery)

## 1. The ML Testing Pyramid
In traditional software, CI tests code logic. In MLOps, CI tests **Code + Data + Models**.

```
         /\     Model Invariance & Directional Tests (Behavioral)
        /  \    -------------------------------------------------
       /    \   Data Validation Tests (Nulls, Data Types, Ranges)
      /      \  -------------------------------------------------
     /________\ Code Unit Tests (Preprocessing logic, Utils)
```

---

## 2. Hands-on: Automated Data & Model Testing Suite


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

def preprocess_input(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    df_clean['income'] = df_clean['income'].fillna(df_clean['income'].median())
    df_clean['age'] = np.clip(df_clean['age'], 18, 100)
    return df_clean

def test_code_unit():
    raw_data = pd.DataFrame({"age": [15, 45, 120], "income": [50000, np.nan, 80000]})
    cleaned = preprocess_input(raw_data)
    
    assert cleaned['income'].isnull().sum() == 0, "❌ Failed: Missing income values remaining!"
    assert cleaned['age'].min() >= 18, "❌ Failed: Age below minimum limit!"
    assert cleaned['age'].max() <= 100, "❌ Failed: Age above maximum limit!"
    print("✅ TEST 1 PASSED: Preprocessing unit tests clean.")

def test_model_invariance():
    X = np.random.rand(100, 2)
    y = (X[:, 0] + X[:, 1] > 1).astype(int)
    model = LogisticRegression().fit(X, y)
    
    base_sample = np.array([[0.8, 0.8]])
    noisy_sample = np.array([[0.801, 0.801]])
    
    pred_base = model.predict(base_sample)[0]
    pred_noisy = model.predict(noisy_sample)[0]
    
    assert pred_base == pred_noisy, "❌ Failed: Model prediction unstable under minimal noise!"
    print("✅ TEST 2 PASSED: Model invariance test clean.")

test_code_unit()
test_model_invariance()


---

## 3. GitHub Actions CI/CD Workflow Template (`.github/workflows/ml_ci.yml`)

```yaml
name: MLOps Continuous Integration

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test-and-validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Set up Python 3.10
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest scikit-learn pandas numpy

      - name: Run Test Suite
        run: |
          pytest tests/
```
